In [10]:
import os, csv, json, glob, shutil, hashlib, zipfile, subprocess, datetime
from collections import Counter
import requests, h5py, numpy as np
from PIL import Image
from kaggle_secrets import UserSecretsClient

USER, EMAIL, REPO = "shashiranjan-jnu", "shashi17_scs@jnu.ac.in", "dpfl-braintumor"
SECRET = "kaggle_dpfl-braintumor"
TODAY = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d")

FIGSHARE_ID, FIGSHARE_VERSION = 1512427, 8
SARTAJ_SLUG    = "brain-tumor-classification-mri"
SARTAJ_URL     = "https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri"
SARTAJ_VERSION = "3"   # from the Data Card, e.g. "v2"
SARTAJ_LICENCE = "MIT"   # from the Data Card
BR35H_SLUG     = "brain-tumor-detection"
BR35H_URL      = "https://www.kaggle.com/datasets/ahmedhamada0/brain-tumor-detection"
BR35H_VERSION  = "12"
BR35H_LICENCE  = "Data files © Original Authors"

WORK = "/kaggle/working"
RAW = f"{WORK}/raw"
MAN = f"{WORK}/manifest"
os.makedirs(RAW, exist_ok=True)
os.makedirs(MAN, exist_ok=True)

token = UserSecretsClient().get_secret(SECRET)

def run(cmd, cwd=None, check=True):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token, "***")
    if check and r.returncode != 0:
        raise RuntimeError(out)
    return r.returncode, out

def file_hash(path, algo="sha256", buf=1 << 20):
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(buf), b""):
            h.update(b)
    return h.hexdigest()

In [11]:
api = f"https://api.figshare.com/v2/articles/{FIGSHARE_ID}/versions/{FIGSHARE_VERSION}"
meta = requests.get(api, timeout=60).json()
FIGSHARE_LICENCE = meta.get("license", {}).get("name", "unknown")
FIGSHARE_URL = f"https://doi.org/{meta.get('doi', '10.6084/m9.figshare.1512427')}"

FS = f"{RAW}/figshare"
os.makedirs(FS, exist_ok=True)
for f in meta["files"]:
    dest = f"{FS}/{f['name']}"
    want = f.get("computed_md5") or f.get("supplied_md5")
    if os.path.exists(dest) and file_hash(dest, "md5") == want:
        print("cached  ", f["name"])
        continue
    with requests.get(f["download_url"], stream=True, timeout=600,
                      headers={"User-Agent": "Mozilla/5.0"}) as r:
        r.raise_for_status()
        with open(dest, "wb") as out:
            for chunk in r.iter_content(1 << 20):
                out.write(chunk)
    assert file_hash(dest, "md5") == want, f"MD5 mismatch for {f['name']}"
    print("verified", f["name"], f["size"])

print("Licence", FIGSHARE_LICENCE, "|", FIGSHARE_URL)

cached   brainTumorDataPublic_1-766.zip
cached   brainTumorDataPublic_1533-2298.zip
cached   brainTumorDataPublic_767-1532.zip
cached   brainTumorDataPublic_2299-3064.zip
cached   cvind.mat
cached   README 2024.txt
Licence CC BY 4.0 | https://doi.org/10.6084/m9.figshare.1512427.v8


In [12]:
MAT = f"{FS}/mat"
os.makedirs(MAT, exist_ok=True)
for z in sorted(glob.glob(f"{FS}/*.zip")):
    with zipfile.ZipFile(z) as zf:
        zf.extractall(MAT)

LABELS = {1: "meningioma", 2: "glioma", 3: "pituitary"}

def read_cj(path):
    with h5py.File(path, "r") as f:
        cj = f["cjdata"]
        label = int(np.array(cj["label"]).squeeze())
        pid = "".join(chr(int(c)) for c in np.array(cj["PID"]).flatten()).strip()
        shape = np.array(cj["image"]).shape
        area = int(np.array(cj["tumorMask"]).sum())
    return label, pid, shape, area

fig_rows = []
for p in sorted(glob.glob(f"{MAT}/**/*.mat", recursive=True)):
    if os.path.basename(p) == "cvind.mat":
        continue
    label, pid, shape, area = read_cj(p)
    fig_rows.append(dict(source="figshare", relpath=os.path.relpath(p, MAT),
                         orig_split="none", label=LABELS[label], patient_id=pid,
                         height=shape[0], width=shape[1], mask_area=area,
                         sha256=file_hash(p)))
print(len(fig_rows), "Figshare slices indexed")

3064 Figshare slices indexed


In [13]:
def find_input(name):
    for root, dirs, _ in os.walk("/kaggle/input"):
        if os.path.basename(root) == name:
            return root
    raise FileNotFoundError(f"Attach {name} through Add Input")

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
SARTAJ_MAP = {"glioma_tumor": "glioma", "meningioma_tumor": "meningioma",
              "pituitary_tumor": "pituitary", "no_tumor": "notumor"}

BR35H_MAP = {"yes": "tumor_unspecified", "no": "notumor", "pred": "unlabelled",
             "br35h-mask-rcnn": "annotated_duplicate"}
SPLITS = ("training", "testing", "train", "test", "val")



def index_images(src, root, cmap):
    out = []
    for p in sorted(glob.glob(f"{root}/**/*", recursive=True)):
        if not p.lower().endswith(IMG_EXT):
            continue
        rel = os.path.relpath(p, root)
        parts = [x.lower() for x in rel.split(os.sep)[:-1]]
        label = next((cmap[x] for x in reversed(parts) if x in cmap), "unknown")
        split = next((x for x in parts if x in SPLITS), "none")
        with Image.open(p) as im:
            w, h = im.size
        out.append(dict(source=src, relpath=rel, orig_split=split, label=label,
                        patient_id="", height=h, width=w, mask_area="",
                        sha256=file_hash(p)))
    return out

sartaj_rows = index_images("sartaj", find_input(SARTAJ_SLUG), SARTAJ_MAP)
br35h_rows = index_images("br35h", find_input(BR35H_SLUG), BR35H_MAP)
print(len(sartaj_rows), "SARTAJ images |", len(br35h_rows), "Br35H images")

3264 SARTAJ images | 3861 Br35H images


In [14]:
index = fig_rows + sartaj_rows + br35h_rows
with open(f"{MAN}/raw_index.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(index[0].keys()))
    w.writeheader()
    w.writerows(index)


CLASSES = ["glioma", "meningioma", "pituitary", "notumor",
           "tumor_unspecified", "unlabelled", "annotated_duplicate", "unknown"]


def counts(src):
    c = Counter(r["label"] for r in index if r["source"] == src)
    return {k: c.get(k, 0) for k in CLASSES}

sources = [
    dict(source="figshare", url=FIGSHARE_URL, version=f"v{FIGSHARE_VERSION}",
         licence=FIGSHARE_LICENCE, redistribution="CHECK", download_date=TODAY,
         n_files=len(fig_rows), **counts("figshare")),
    dict(source="sartaj", url=SARTAJ_URL, version=SARTAJ_VERSION,
         licence=SARTAJ_LICENCE, redistribution="CHECK", download_date=TODAY,
         n_files=len(sartaj_rows), **counts("sartaj")),
    dict(source="br35h", url=BR35H_URL, version=BR35H_VERSION,
         licence=BR35H_LICENCE, redistribution="CHECK", download_date=TODAY,
         n_files=len(br35h_rows), **counts("br35h")),
]
with open(f"{MAN}/sources.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(sources[0].keys()))
    w.writeheader()
    w.writerows(sources)

fc = counts("figshare")
pids = {r["patient_id"] for r in fig_rows}
dup = Counter(r["sha256"] for r in index)
checks = {
    "figshare_total_3064": len(fig_rows) == 3064,
    "figshare_classes_708_1426_930": (fc["meningioma"], fc["glioma"], fc["pituitary"]) == (708, 1426, 930),
    "figshare_patients": len(pids),
    "figshare_every_slice_has_pid": all(r["patient_id"] for r in fig_rows),
    "figshare_every_slice_has_mask": all(r["mask_area"] > 0 for r in fig_rows),
    "no_unknown_labels": all(r["label"] != "unknown" for r in index),
    "exact_duplicate_files": sum(v - 1 for v in dup.values() if v > 1),
    "versions_and_licences_filled": "FILL" not in (SARTAJ_VERSION, SARTAJ_LICENCE,
                                                   BR35H_VERSION, BR35H_LICENCE),
}
json.dump(checks, open(f"{MAN}/acquisition_checks.json", "w"), indent=2)
for k, v in checks.items():
    print(f"{k:35s} {v}")
for s in sources:
    print(s["source"], {k: s[k] for k in CLASSES if s[k]})

figshare_total_3064                 True
figshare_classes_708_1426_930       True
figshare_patients                   233
figshare_every_slice_has_pid        True
figshare_every_slice_has_mask       True
no_unknown_labels                   True
exact_duplicate_files               1274
versions_and_licences_filled        True
figshare {'glioma': 1426, 'meningioma': 708, 'pituitary': 930}
sartaj {'glioma': 926, 'meningioma': 937, 'pituitary': 901, 'notumor': 500}
br35h {'notumor': 1500, 'tumor_unspecified': 1500, 'unlabelled': 60, 'annotated_duplicate': 801}


In [ ]:
import pandas as pd
df = pd.DataFrame(index)

# A. Folder structure of Br35H, two levels deep
root = find_input(BR35H_SLUG)
for r, ds, fs in os.walk(root):
    depth = r[len(root):].count(os.sep)
    if depth <= 2:
        exts = Counter(os.path.splitext(x)[1].lower() for x in fs)
        print(f"{r[len(root):] or '/':40s} dirs={sorted(ds)[:8]} files={dict(exts)}")

# B. Where the unknown images live
unk = df[df.label == "unknown"]
print("\nUnknown images by folder")
print(unk.relpath.map(lambda p: os.path.dirname(p)).value_counts().head(20))

# C. Duplicate groups by source combination and label combination
d = df[df.duplicated("sha256", keep=False)]
g = d.groupby("sha256").agg(
    sources=("source", lambda s: "+".join(sorted(set(s)))),
    labels=("label", lambda s: "+".join(sorted(set(s)))),
    splits=("orig_split", lambda s: "+".join(sorted(set(s)))),
    n=("relpath", "size"))
print("\nDuplicate groups by sources and labels")
print(g.groupby(["sources", "labels"]).size().sort_values(ascending=False).to_string())

# D. The two dangerous cases
real_labels = {"glioma", "meningioma", "pituitary", "notumor"}
conflict = g[g.labels.map(lambda s: len(set(s.split("+")) & real_labels) > 1)]
leak = g[(g.sources == "sartaj") & g.splits.str.contains("training") & g.splits.str.contains("testing")]
print("\nGroups with conflicting real labels", len(conflict))
print("SARTAJ groups present in both training and testing", len(leak))
print(conflict.head(10).to_string())

In [ ]:
os.chdir(WORK)
run(["rm", "-rf", REPO])
auth = f"https://{token}@github.com/{USER}/{REPO}.git"
clean = f"https://github.com/{USER}/{REPO}.git"

run(["git", "clone", "-q", "-b", "main", auth, REPO])
run(["git", "remote", "set-url", "origin", clean], cwd=REPO)

files = ["raw_index.csv", "sources.csv", "acquisition_checks.json"]
if os.path.exists(f"{MAN}/exact_duplicate_summary.csv"):
    files.append("exact_duplicate_summary.csv")
for name in files:
    shutil.copy(f"{MAN}/{name}", f"{REPO}/data_manifest/{name}")

lab = f"{REPO}/docs/lab_notebook.md"
marker = f"## {TODAY} Step 1.3 data acquisition"
if marker not in open(lab).read():
    with open(lab, "a") as f:
        f.write(f"\n{marker}\n")
        for s in sources:
            f.write(f"- {s['source']} {s['version']}, {s['n_files']} files, licence {s['licence']}\n")
        f.write(f"- Figshare patients {checks['figshare_patients']}, "
                f"exact duplicate files across sources {checks['exact_duplicate_files']}\n")

run(["git", "config", "user.name", USER], cwd=REPO)
run(["git", "config", "user.email", EMAIL], cwd=REPO)
run(["git", "add", "-A"], cwd=REPO)
_, status = run(["git", "status", "--porcelain"], cwd=REPO)
if status.strip():
    run(["git", "commit", "-q", "-m", "Step 1.3 raw data manifests"], cwd=REPO)
    run(["git", "push", "-q", auth, "main"], cwd=REPO)
    print("Manifests pushed to main.")
else:
    print("Nothing new to commit.")

shutil.rmtree(f"{WORK}/{REPO}")
if os.path.exists(MAT):
    shutil.rmtree(MAT)
print("Working copy removed.")